In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()   #carrega a variável de ambiente GOOGLE_API_KEY do arquivo .env

model = init_chat_model("openai:gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([("user", "Conte uma curiosidade sobre {tema}.")])

chain = prompt | model

chain.invoke({"tema": "o oceano"})

In [ ]:
for c in chain.stream({"tema": "o oceano"}):
    print(c.text, end="")

In [ ]:
chain.batch([{"tema": "Marte"}, {"tema": "abelhas"}]) # em paralelo

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template([
    ("system", "Você é um assistente conciso. Responda em uma frase."),
    ("user", "Explique o que é {conceito}."),
])

# prompt -> model -> parser de string
chain = prompt | model | StrOutputParser()

texto = chain.invoke({"conceito": "machine learning"})
print(texto)
print(type(texto)) # <class 'langchain_core.messages.base.TextAccessor'>

In [ ]:
from pydantic import BaseModel, Field

class Resumo(BaseModel):
    """Resumo estruturado de um texto."""
    titulo: str = Field(description="Um título curto para o conteúdo")
    pontos: list[str] = Field(description="3 a 5 pontos principais")
    
# Cria uma versão do modelo que devolve um objeto Resumo
modelo_estruturado = model.with_structured_output(Resumo)

resultado = modelo_estruturado.invoke("Resuma os beneficios da energia solar.")

print(type(resultado))      # <class '....Resumo'> (objeto Pydantic, não string)
print(resultado.titulo)
for p in resultado.pontos:
    print("-", p)

In [ ]:
chain = prompt | modelo_estruturado
resultado = chain.invoke({"conceito": "Banco de Dados Vetoriais"})

In [ ]:
for ponto in resultado.pontos:
    print("()", ponto)

In [ ]:
from langchain_core.runnables import (
    RunnablePassthrough, # passa a entrada adiante, sem alterar
    RunnableParallel, # roda vários ramos em paralelo
    RunnableLambda, # transforma uma função sua em Runnable
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

prompt_resumo = ChatPromptTemplate.from_messages([("user", "Resuma em uma frase: {texto}")])
resumo_chain = (
    prompt_resumo | model | StrOutputParser()
)

prompt_sentimento = ChatPromptTemplate.from_messages([
    ("user", "Classifique o sentimento (positivo/negativo/neutro): {texto}")
])

sentimento_chain = (
    prompt_sentimento | model | StrOutputParser()
)

# Os dois ramos recebem a MESMA entrada e rodam em paralelo
analise = RunnableParallel(
    resumo = resumo_chain,
    sentimento = sentimento_chain,
)

resultado = analise.invoke(
    {"texto": "O atendimento foi excelente e a entrega chegou antes do prazo."}
)

print(resultado["resumo"])
print(resultado["sentimento"])
# -> {"resumo": "...", "sentimento": "positivo"}

In [ ]:
from langchain_core.runnables import RunnableLambda

# Transforma {"texto": "..."} em {"texto": "...", "tamanho": N}
adicionar_tamanho = RunnableLambda(
    lambda x: {"texto": x["texto"], "tamanho": len(x["texto"])}
)

adicionar_tamanho.invoke({"texto": "olá mundo"}) # -> {"texto": "olá mundo", "tamanho": 9}

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Etapa 1: gera um slogan a partir de um produto
gerar_slogan = (
    ChatPromptTemplate.from_messages([
        ("system", "Você é um redator publicitário."),
        ("user", "Crie um slogan curto para: {produto}"),
    ])
    | model | StrOutputParser()
)

# Etapa 2: traduz o que receber. Note a variável {slogan}.
traduzir = (
    ChatPromptTemplate.from_messages([
        ("user", "Traduza para inglês, mantendo o impacto: {slogan}")
    ])
    | model | StrOutputParser()
)

# A saída da etapa 1(string) precisa virar {"slogan": <texto>} para a etapa 2.
# RunnableLambda faz essa "ponte" entre as etapas
from langchain_core.runnables import RunnableLambda
ponte = RunnableLambda(lambda slogan: {"slogan": slogan})

fluxo = gerar_slogan | ponte | traduzir

resultado = fluxo.invoke({"produto": "uma cafeteira inteligente"})
print(resultado) # slogan já gerado e traduzido para inglês